# Notebook 01.2 — Aquisição da infraestrutura viária e edificações (Overture Maps)

**Projeto:** Acessibilidade Geográfica às UBS de Teresina: roteiro computacional AE2SFCA  
**Programa:** MAPEPROF - Mestrado Profissional em Planejamento Urbano e Regional / IFPI  
**Autor:** Felipe Ramos Dantas  
**Orientador:** Prof. Dr. Antonio Joaquim da Silva  
**Coorientador:** Prof. Dr. Reurysson Chagas de Sousa Morais  
**Repositório:** https://github.com/felipedantas-pi/mapeprof-accessibility-ubs  
**Última atualização:** 2026-05-06

## Objetivo

Utilizar a área de estudo pré-processada no **Notebook 01.1** (Zona Urbana + *buffer* de 5km) para extrair da **Overture Maps Foundation** os dados de infraestrutura urbana de Teresina. Serão obtidos os grafos da malha viária (*segment* e *connector*) e as pegadas de construção (*building*), servindo de base para a rede geométrica do modelo AE2SFCA.

## Saída

| Arquivo | Destino | Descrição |
|---|---|---|
| `zonaUrbanaReal_5km_segmentos.parquet` | NB 02.1 | Segmentos viários brutos (arestas do grafo) |
| `zonaUrbanaReal_5km_conectores.parquet` | NB 02.1 | Conectores viários brutos (nós do grafo) |
| `teresina_building_utm.geojson` | NB 02.4 | Pegadas de construção (proxy espacial de domicílios) |

## Pré-requisitos
- O Notebook **01.1** deve ter sido executado previamente para gerar os recortes territoriais no diretório `../dados/brutos/ibge/`.
- Conexão de internet estável.

---

In [1]:
# ── 1. IMPORTAÇÕES E CONFIGURAÇÕES GLOBAIS ───────────────────────────────────

import geopandas as gpd
import city2graph as c2g
from overturemaps import geodataframe as overture_gdf
from pathlib import Path

# Sistemas de Referência de Coordenadas (SRC)
SRC_METRICO    = "31983"   # SIRGAS 2000 / UTM Zona 23S
SRC_GEOGRAFICO = "4326"    # WGS 84 (Exigido pela API da Overture Maps)

# Diretórios de integração
DIR_IBGE = Path("../dados/brutos/ibge")
DIR_OVERTURE = Path("../dados/brutos/overture")
DIR_OVERTURE.mkdir(parents=True, exist_ok=True)

print("✅ Dependências carregadas e diretórios configurados.")
print(f"city2graph version: {c2g.__version__ if hasattr(c2g, '__version__') else 'development'}")

✅ Dependências carregadas e diretórios configurados.
city2graph version: 0.3.1


## 2. Carregamento da Área de Estudo

A API da Overture Maps realiza requisições espaciais utilizando coordenadas geográficas padrão (Latitude/Longitude). Portanto, o primeiro passo é importar os polígonos métricos de Teresina gerados no notebook anterior e reprojetá-los para `EPSG:4326` (WGS 84) antes de iniciar o *download*.

In [2]:
# ── 2. IMPORTAÇÃO DOS DADOS DO NOTEBOOK 01.1 ─────────────────────────────────

print("🗺️ Carregando recortes espaciais gerados no Notebook 01.1...")

# Carrega os polígonos projetados em UTM (EPSG:31983)
gdf_teresina = gpd.read_parquet(DIR_IBGE / "teresina.parquet")
gdf_zonaUrbana_5km = gpd.read_parquet(DIR_IBGE / "teresina_zonaUrbanaClip_utm.parquet")

# Reprojeta estritamente para WGS 84 para comunicação com os servidores da Overture
gdf_teresina_wgs = gdf_teresina.to_crs(SRC_GEOGRAFICO)
gdf_zonaUrbana_wgs = gdf_zonaUrbana_5km.to_crs(SRC_GEOGRAFICO)

print("✅ Dados carregados e convertidos para WGS 84 com sucesso.")

🗺️ Carregando recortes espaciais gerados no Notebook 01.1...
✅ Dados carregados e convertidos para WGS 84 com sucesso.


## 3. Aquisição da Malha Viária (Grafos base)

Os subconjuntos `segment` (arestas da rua) e `connector` (nós de interseção) são estruturados pela biblioteca `city2graph`. A versão da base (*release*) é fixada para garantir a reprodutibilidade metodológica da dissertação, evitando que atualizações futuras da plataforma Overture alterem a topologia da rede estudada.

In [3]:
# ── 3. DOWNLOAD SEGMENTOS E CONECTORES ───────────────────────────────────────

subdatasets = ["segment", "connector"]
RELEASE_OVERTURE = '2026-05-20.0'

print(f"🛣️ Extraindo malha viária (Release {RELEASE_OVERTURE}). Isso pode levar alguns minutos...")

dados_viarios = c2g.load_overture_data(
    area=gdf_zonaUrbana_wgs,      # O polígono de busca (área de estudo já com buffer)
    types=subdatasets,
    output_dir=DIR_OVERTURE,
    prefix='zonaUrbanaReal_5km_',
    save_to_file=False,           # Mantemos em memória para reprojetar antes de salvar
    return_data=True,
    release=RELEASE_OVERTURE,
    use_stac=False
)

print("🔄 Reprojetando para UTM e salvando grafos...")
segments_metric = dados_viarios['segment'].to_crs(SRC_METRICO)
connectors_metric = dados_viarios['connector'].to_crs(SRC_METRICO)

# 1. Exportando GeoJSON (Formato Padrão/Seguro para listas aninhadas do algoritmo)
segments_metric.to_file(DIR_OVERTURE / "zonaUrbanaReal_5km_segmentos.geojson", driver="GeoJSON")
connectors_metric.to_file(DIR_OVERTURE / "zonaUrbanaReal_5km_conectores.geojson", driver="GeoJSON")
print("✅ Arquivos GeoJSON gerados com sucesso.")

# 2. Exportando Parquet (Formato comprimido para análise local de alta performance)
segments_metric.to_parquet(DIR_OVERTURE / "zonaUrbanaReal_5km_segmentos.parquet", index=False)
connectors_metric.to_parquet(DIR_OVERTURE / "zonaUrbanaReal_5km_conectores.parquet", index=False)
print("✅ Arquivos Parquet de backup gerados com sucesso.")

🛣️ Extraindo malha viária (Release 2026-05-20.0). Isso pode levar alguns minutos...
🔄 Reprojetando para UTM e salvando grafos...
✅ Arquivos GeoJSON gerados com sucesso.
✅ Arquivos Parquet de backup gerados com sucesso.


## 4. Aquisição de Edificações (Building Footprints)

Devido à alta densidade do *dataset* de edificações, utiliza-se diretamente o método `geodataframe` da biblioteca `overturemaps-py` em conjunto com um *Bounding Box* (retângulo envolvente) do município. As feições baixadas servirão como *proxy* para a distribuição micrométrica da população de Teresina.

In [7]:
# ── 4. DOWNLOAD DAS PEGADAS DE CONSTRUÇÃO ────────────────────────────────────

print("🏠 Calculando limites do município para extração de edificações...")

# Extrai o retângulo envolvente (Bounding Box) automático de Teresina [minX, minY, maxX, maxY]
bounds = gdf_teresina_wgs.total_bounds
bbox_teresina = (bounds[0], bounds[1], bounds[2], bounds[3])

print(f"📡 Solicitando base de 'buildings' para a Bounding Box: {bbox_teresina}")
# Chamada direta e segura à biblioteca nativa overturemaps
gdf_building = overture_gdf("building", bbox=bbox_teresina, release=RELEASE_OVERTURE)

# Confirma o CRS geográfico inicial
gdf_building.set_crs("EPSG:4326", inplace=True)

print(f"✅ Download concluído. Total de edificações mapeadas: {len(gdf_building):,}")

# Exportação (Pode ser alterada para .parquet se o software QGIS suportar, 
# mas mantemos o .geojson original caso exija interoperabilidade externa imediata)
print("💾 Reprojetando para UTM e exportando arquivo...")
caminho_buildings = DIR_OVERTURE / "teresina_building_utm.geojson"
gdf_building.to_crs(SRC_METRICO).to_file(caminho_buildings, driver="GeoJSON")

print(f"🎉 Processo concluído! Arquivo salvo em: {caminho_buildings}")

🏠 Calculando limites do município para extração de edificações...
📡 Solicitando base de 'buildings' para a Bounding Box: (-42.97066660000001, -5.586602499999965, -42.5989726999999, -4.78627779999994)
✅ Download concluído. Total de edificações mapeadas: 521,014
💾 Reprojetando para UTM e exportando arquivo...
🎉 Processo concluído! Arquivo salvo em: ..\dados\brutos\overture\teresina_building_utm.geojson


5. Aquisição de Unidades Básicas de Saúde (UBS)



In [ ]:
# CNES health facilities
from pysus import cnes

df = cnes(state="PI", year=2025, month=12)
df.head()

In [ ]:
df.to_csv("../dados/brutos/cnes_pysus_pi.csv", sep=';', encoding='utf-8')